In [10]:
# ----------------------------------------
# Project 05 - Microsoft Fabric Analytics Platform
# Notebook 06 - Full Silver Transformation
# ----------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table("bronze.meter_readings")

print("Full-scale Silver transformation initialised.")

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 12, Finished, Available, Finished, False)

Full-scale Silver transformation initialised.


In [11]:
standardised_df = (
    bronze_df
    .select(
        F.col("LCLid").alias("HouseholdID"),
        F.col("stdorToU").alias("TariffType"),
        F.col("DateTime").alias("ReadingTimestampRaw"),
        F.col("ConsumptionKWhRaw").alias("ConsumptionKWhRaw"),
        F.col("SourceFileName"),
        F.col("SourceFilePath"),
        F.col("IngestionTimestamp"),
        F.col("SourceSystem")
    )
)

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 13, Finished, Available, Finished, False)

In [12]:
typed_df = (
    standardised_df
    .withColumn(
        "ReadingTimestamp",
        F.to_timestamp("ReadingTimestampRaw")
    )
    .withColumn(
        "ConsumptionKWh",
        F.col("ConsumptionKWhRaw").cast("double")
    )
)

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 14, Finished, Available, Finished, False)

In [13]:
validated_df = (
    typed_df
    .withColumn(
        "IsMissingHousehold",
        F.col("HouseholdID").isNull() |
        (F.trim(F.col("HouseholdID")) == "")
    )
    .withColumn(
        "IsInvalidTimestamp",
        F.col("ReadingTimestamp").isNull()
    )
    .withColumn(
        "IsInvalidConsumption",
        F.col("ConsumptionKWh").isNull() |
        (F.col("ConsumptionKWh") < 0)
    )
    .withColumn(
        "IsInvalidTariff",
        F.col("TariffType").isNull() |
        (~F.col("TariffType").isin("Std", "ToU"))
    )
)

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 15, Finished, Available, Finished, False)

In [14]:
validated_df = (
    validated_df
    .withColumn(
        "RejectionReason",
        F.when(
            F.col("IsMissingHousehold"),
            F.lit("MISSING_HOUSEHOLD")
        )
        .when(
            F.col("IsInvalidTimestamp"),
            F.lit("INVALID_TIMESTAMP")
        )
        .when(
            F.col("IsInvalidConsumption"),
            F.lit("INVALID_CONSUMPTION")
        )
        .when(
            F.col("IsInvalidTariff"),
            F.lit("INVALID_TARIFF")
        )
        .otherwise(F.lit(None))
    )
)

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 16, Finished, Available, Finished, False)

In [15]:
rejected_df = (
    validated_df
    .filter(F.col("RejectionReason").isNotNull())
)

valid_df = (
    validated_df
    .filter(F.col("RejectionReason").isNull())
)

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 17, Finished, Available, Finished, False)

In [16]:
dedup_window = (
    Window
    .partitionBy(
        "HouseholdID",
        "ReadingTimestamp"
    )
    .orderBy(
        F.col("SourceFileName").asc(),
        F.col("ConsumptionKWh").asc_nulls_last(),
        F.col("TariffType").asc_nulls_last()
    )
)

ranked_df = (
    valid_df
    .withColumn(
        "DuplicateRank",
        F.row_number().over(dedup_window)
    )
)

duplicates_removed_df = (
    ranked_df
    .filter(F.col("DuplicateRank") > 1)
)

silver_base_df = (
    ranked_df
    .filter(F.col("DuplicateRank") == 1)
)

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 18, Finished, Available, Finished, False)

In [17]:
silver_df = (
    silver_base_df
    .withColumn(
        "ReadingDate",
        F.to_date("ReadingTimestamp")
    )
    .withColumn(
        "ReadingYear",
        F.year("ReadingTimestamp")
    )
    .withColumn(
        "ReadingMonth",
        F.month("ReadingTimestamp")
    )
    .withColumn(
        "ReadingDay",
        F.dayofmonth("ReadingTimestamp")
    )
    .withColumn(
        "ReadingHour",
        F.hour("ReadingTimestamp")
    )
    .withColumn(
        "DayOfWeek",
        F.date_format("ReadingTimestamp", "EEEE")
    )
)

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 19, Finished, Available, Finished, False)

In [18]:
silver_final_df = (
    silver_df
    .select(
        "HouseholdID",
        "TariffType",
        "ReadingTimestamp",
        "ConsumptionKWh",
        "ReadingDate",
        "ReadingYear",
        "ReadingMonth",
        "ReadingDay",
        "ReadingHour",
        "DayOfWeek",
        "SourceFileName",
        "IngestionTimestamp",
        "SourceSystem"
    )
)

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 20, Finished, Available, Finished, False)

In [19]:
rejected_output_df = (
    rejected_df
    .select(
        "HouseholdID",
        "TariffType",
        "ReadingTimestampRaw",
        "ConsumptionKWhRaw",
        "SourceFileName",
        "SourceFilePath",
        "IngestionTimestamp",
        "SourceSystem",
        "RejectionReason"
    )
)

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 21, Finished, Available, Finished, False)

In [20]:
spark.sql("DROP TABLE IF EXISTS silver.meter_readings")

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 22, Finished, Available, Finished, False)

DataFrame[]

In [21]:
(
    silver_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.meter_readings")
)

print("silver.meter_readings written successfully.")

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 23, Finished, Available, Finished, False)

silver.meter_readings written successfully.


In [22]:
spark.sql("DROP TABLE IF EXISTS silver.rejected_meter_readings")

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 24, Finished, Available, Finished, False)

DataFrame[]

In [23]:
(
    rejected_output_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.rejected_meter_readings")
)

print("silver.rejected_meter_readings written successfully.")

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 25, Finished, Available, Finished, False)

silver.rejected_meter_readings written successfully.


In [24]:
bronze_count = bronze_df.count()
silver_count = spark.table("silver.meter_readings").count()
rejected_count = spark.table("silver.rejected_meter_readings").count()
duplicates_removed_count = duplicates_removed_df.count()

print("FULL SILVER RECONCILIATION")
print("-" * 50)
print(f"Bronze rows:               {bronze_count:,}")
print(f"Rejected rows:             {rejected_count:,}")
print(f"Duplicates removed:        {duplicates_removed_count:,}")
print(f"Silver rows:               {silver_count:,}")
print(
    f"Reconciliation difference: "
    f"{bronze_count - rejected_count - duplicates_removed_count - silver_count:,}"
)

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 26, Finished, Available, Finished, False)

FULL SILVER RECONCILIATION
--------------------------------------------------
Bronze rows:               167,932,474
Rejected rows:             5,560
Duplicates removed:        115,453
Silver rows:               167,811,461
Reconciliation difference: 0


In [25]:
print("REJECTION SUMMARY")
print("-" * 50)

spark.table("silver.rejected_meter_readings") \
    .groupBy("RejectionReason") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

StatementMeta(, 1b3bb298-0cde-4425-a3fa-521edb5d45ef, 27, Finished, Available, Finished, False)

REJECTION SUMMARY
--------------------------------------------------
+-------------------+-----+
|RejectionReason    |count|
+-------------------+-----+
|INVALID_CONSUMPTION|5560 |
+-------------------+-----+



In [ ]:
silver_check_df = spark.table("silver.meter_readings")

print("SILVER QA")
print("-" * 50)

duplicate_keys = (
    silver_check_df
    .groupBy(
        "HouseholdID",
        "ReadingTimestamp"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_consumption = (
    silver_check_df
    .filter(
        F.col("ConsumptionKWh").isNull() |
        (F.col("ConsumptionKWh") < 0)
    )
    .count()
)

invalid_tariff = (
    silver_check_df
    .filter(
        ~F.col("TariffType").isin("Std", "ToU")
    )
    .count()
)

print(f"Duplicate business keys: {duplicate_keys:,}")
print(f"Invalid consumption rows: {invalid_consumption:,}")
print(f"Invalid tariff rows:      {invalid_tariff:,}")